# MDP Solving

## Learning Objectives

By the end of this notebook, you will be able to:
- explain what it means to *solve* a small Markov Decision Process
- evaluate a fixed policy using repeated Bellman updates
- improve a policy by choosing better actions from state values
- run policy iteration on a tiny grid world

## Prerequisites

Before starting:
- complete `01_mdp_example.ipynb`
- be comfortable with states, actions, rewards, and terminal states
- know basic Python loops and NumPy arrays

## Why this notebook matters

In the previous notebook, you defined an MDP. In this notebook, you take the next step:

1. start with a policy
2. estimate how good that policy is
3. improve it
4. repeat until the policy stops changing

That process is the bridge between *understanding an MDP* and *computing a solution*.


## Lesson Brief

This lesson moves from **describing** an MDP to **solving** it.

Students will see how policy evaluation and policy improvement turn an MDP from a problem description into a decision procedure.

Why this matters: this is the first time the class sees how RL-style reasoning can produce better behavior from the same environment.

This lesson builds directly on the MDP structure from the previous notebook.

## What does “solving an MDP” mean?

For a small MDP, solving means answering two questions:

- **How good is each state?**
  This is the value function.
- **What action should the agent take in each state?**
  This is the policy.

In this notebook, we will use a tiny deterministic grid world so every update is easy to follow.

## Teaching flow

- First: build the environment
- Then: evaluate one fixed policy
- Then: improve the policy
- Finally: run full policy iteration

## Checkpoint Before Coding

Before you run the code, make sure you can answer these questions:

1. What is a **policy**?
2. What is the difference between a **value function** and a **policy**?
3. Why can the best action depend on the *future* reward, not just the immediate reward?

If any of these are unclear, go back to `01_mdp_example.ipynb` before continuing.


In [1]:
import numpy as np

np.set_printoptions(precision=2, suppress=True)

print("=" * 70)
print("MDP Solving with Policy Evaluation and Policy Iteration")
print("=" * 70)

# Small 3x3 grid world
N_ROWS, N_COLS = 3, 3
N_STATES = N_ROWS * N_COLS
ACTIONS = ["up", "right", "down", "left"]
ACTION_TO_DELTA = {
    0: (-1, 0),
    1: (0, 1),
    2: (1, 0),
    3: (0, -1),
}
ACTION_TO_ARROW = {0: "↑", 1: "→", 2: "↓", 3: "←"}

GOAL_STATE = 8
PIT_STATE = 6
TERMINAL_STATES = {GOAL_STATE, PIT_STATE}
GAMMA = 0.90


def to_pos(state):
    return divmod(state, N_COLS)


def to_state(row, col):
    return row * N_COLS + col


def transition(state, action):
    """Deterministic transition for a tiny teaching environment."""
    if state in TERMINAL_STATES:
        return state, 0.0

    row, col = to_pos(state)
    d_row, d_col = ACTION_TO_DELTA[action]
    next_row = min(max(row + d_row, 0), N_ROWS - 1)
    next_col = min(max(col + d_col, 0), N_COLS - 1)
    next_state = to_state(next_row, next_col)

    if next_state == GOAL_STATE:
        return next_state, 10.0
    if next_state == PIT_STATE:
        return next_state, -10.0
    return next_state, -1.0


def print_state_labels():
    print("\nState layout:")
    for r in range(N_ROWS):
        row = []
        for c in range(N_COLS):
            s = to_state(r, c)
            if s == GOAL_STATE:
                row.append(" G ")
            elif s == PIT_STATE:
                row.append(" P ")
            else:
                row.append(f" {s} ")
        print(" ".join(row))


def print_values(values):
    print("\nState values:")
    for r in range(N_ROWS):
        row = []
        for c in range(N_COLS):
            s = to_state(r, c)
            if s == GOAL_STATE:
                row.append("  G   ")
            elif s == PIT_STATE:
                row.append("  P   ")
            else:
                row.append(f"{values[s]:6.2f}")
        print(" ".join(row))


def print_policy(policy):
    print("\nPolicy:")
    for r in range(N_ROWS):
        row = []
        for c in range(N_COLS):
            s = to_state(r, c)
            if s == GOAL_STATE:
                row.append(" G ")
            elif s == PIT_STATE:
                row.append(" P ")
            else:
                row.append(f" {ACTION_TO_ARROW[policy[s]]} ")
        print(" ".join(row))


print_state_labels()
print("\nSample transition: state 0 + action 'right' ->", transition(0, 1))
print("Sample transition: state 4 + action 'down'  ->", transition(4, 2))

MDP Solving with Policy Evaluation and Policy Iteration

State layout:
 0   1   2 
 3   4   5 
 P   7   G 

Sample transition: state 0 + action 'right' -> (1, -1.0)
Sample transition: state 4 + action 'down'  -> (7, -1.0)


## Part 1: Policy Evaluation

We start with a simple hand-written policy and ask:

> If the agent follows this policy forever, how good is each state?

This is called **policy evaluation**.

In this example:
- the agent gets `+10` for reaching the goal
- the agent gets `-10` for falling into the pit
- every normal move costs `-1`

That means a good policy should move toward the goal while avoiding the pit.

In [2]:
# A simple fixed policy: try to move right, otherwise move down.
fixed_policy = np.array([
    1, 1, 2,
    1, 1, 2,
    1, 1, 0,
])


def evaluate_policy(policy, gamma=GAMMA, theta=1e-6):
    values = np.zeros(N_STATES)
    deltas = []

    while True:
        delta = 0.0
        for state in range(N_STATES):
            if state in TERMINAL_STATES:
                continue
            action = policy[state]
            next_state, reward = transition(state, action)
            new_value = reward + gamma * values[next_state]
            delta = max(delta, abs(new_value - values[state]))
            values[state] = new_value
        deltas.append(delta)
        if delta < theta:
            break

    return values, deltas


values_fixed, deltas_fixed = evaluate_policy(fixed_policy)

print("Fixed policy arrows:")
print_policy(fixed_policy)
print_values(values_fixed)
print("\nNumber of policy-evaluation sweeps:", len(deltas_fixed))
print("Last Bellman update size:", deltas_fixed[-1])

print("\nCheckpoint:")
print("- States near the goal should have higher values.")
print("- States near the pit should have lower values.")
print("- The values depend on both immediate reward and future reward.")

Fixed policy arrows:

Policy:
 →   →   ↓ 
 →   →   ↓ 
 P   →   G 

State values:
  4.58   6.20   8.00
  6.20   8.00  10.00
  P     10.00   G   

Number of policy-evaluation sweeps: 5
Last Bellman update size: 0.0

Checkpoint:
- States near the goal should have higher values.
- States near the pit should have lower values.
- The values depend on both immediate reward and future reward.


## Part 2: Policy Improvement and Policy Iteration

Policy evaluation tells us how good the current policy is.

Policy improvement asks a second question:

> If I know the values, which action looks best from each state?

If we repeatedly **evaluate** and then **improve**, we get **policy iteration**.

### Common misconception

A policy is **not** the same thing as the value function.

- The **policy** says what to do.
- The **value function** says how good a state is if you follow a policy.

In [ ]:
# Policy iteration = evaluate -> improve -> repeat
def improve_policy(values, gamma=GAMMA):
    policy = np.zeros(N_STATES, dtype=int)
    for state in range(N_STATES):
        if state in TERMINAL_STATES:
            continue
        action_returns = []
        for action in range(len(ACTIONS)):
            next_state, reward = transition(state, action)
            action_returns.append(reward + gamma * values[next_state])
        policy[state] = int(np.argmax(action_returns))
    return policy


def policy_iteration(initial_policy=None, gamma=GAMMA, theta=1e-6):
    if initial_policy is None:
        policy = np.zeros(N_STATES, dtype=int)
    else:
        policy = initial_policy.copy()

    rounds = 0
    while True:
        rounds += 1
        values, _ = evaluate_policy(policy, gamma=gamma, theta=theta)
        improved_policy = improve_policy(values, gamma=gamma)
        if np.array_equal(policy, improved_policy):
            return policy, values, rounds
        policy = improved_policy


optimal_policy, optimal_values, rounds = policy_iteration(fixed_policy)

print("Optimal policy after policy iteration:")
print_policy(optimal_policy)
print_values(optimal_values)
print(f"\nPolicy iteration converged in {rounds} improvement rounds.")

print("\nReflection questions:")
print("1. Why does the policy near the pit avoid that state?")
print("2. Why are values close to the goal higher?")
print("3. What changed between the fixed policy and the improved policy?")

print("\nSummary:")
print("- Policy evaluation estimates state values for one policy.")
print("- Policy improvement chooses better actions from those values.")
print("- Policy iteration alternates both steps until the policy stops changing.")
print("- Next notebook: value iteration solves the same kind of problem more directly.")

## Closing Takeaway

**Teaching takeaway:** Once an MDP is defined, policy evaluation and policy iteration let us measure decisions and improve them step by step.

**If students remember one idea:** Solving an MDP is not magic. It is repeated reasoning about expected future reward under a policy.

**Quick check before moving on:**
- Can you explain the difference between evaluating a policy and improving a policy?
- Can you describe why repeated updates eventually reveal a better decision rule?

**Bridge to the next step:** The next lesson shows a closely related idea: value iteration, which combines evaluation and improvement in one update loop.
